In [ ]:
# Ejecutar desde la raíz del proyecto o desde integraciones/krayin.
from pathlib import Path

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "ML" / "Datasets" / "train.csv").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Abra el notebook dentro de la carpeta del proyecto.")
ML_DIR = PROJECT_ROOT / "ML"
INTEGRATION_DIR = PROJECT_ROOT / "integraciones" / "krayin"

import json
import mysql.connector
import pandas as pd

# 1. Cargar el dataset procesado
df = pd.read_csv(INTEGRATION_DIR / "predicciones_crm_churn_perfecto.csv")

# 2. Conexión a la base de datos MySQL (laravel-crm)
db = mysql.connector.connect(
    host="127.0.0.1",
    user="root",
    password="",  # Contraseña vacía por defecto en XAMPP
    database="laravel-crm",
)
cursor = db.cursor()

print("Cargando registros a la base de datos 'laravel-crm'...")

for idx, row in df.iterrows():
    # Estructuras JSON para la tabla 'persons'
    email_val = json.dumps(
        [{"value": f"cliente_{row['customer_id']}@ejemplo.com", "label": "work"}]
    )
    phone_val = json.dumps([{"value": f"900{idx+1:06d}", "label": "work"}])

    # 3. Insertar registro principal en 'persons'
    sql_person = """
        INSERT INTO persons (name, emails, contact_numbers, user_id, created_at, updated_at)
        VALUES (%s, %s, %s, %s, NOW(), NOW())
    """
    cursor.execute(
        sql_person, (str(row["customer_id"]), email_val, phone_val, 1)
    )
    person_id = cursor.lastrowid

    # 4. Mapeo de Atributos Personalizados
    # 61: customer_id | 62: frecuencia_compras | 63: monto_total_gastado | 64: probabilidad_churn | 65: categoria_riesgo
    atributos = [
        (61, str(row["customer_id"]), "text_value"),
        (62, str(row["frecuencia_compras"]), "text_value"),
        (63, str(row["monto_total_gastado"]), "float_value"),
        (64, str(row["probabilidad_churn"]), "text_value"),
        (65, str(row["categoria_riesgo"]), "text_value"),
    ]

    # Insertar valores en 'attribute_values' (sin columnas created_at/updated_at)
    for attr_id, val, col_type in atributos:
        sql_attr = f""" 
            INSERT INTO attribute_values (entity_type, entity_id, attribute_id, {col_type})
            VALUES ('persons', %s, %s, %s)
        """
        cursor.execute(sql_attr, (person_id, attr_id, val))

# Guardar cambios en la base de datos
db.commit()
print(
    f"¡Proceso finalizado! Se insertaron {len(df)} contactos con sus atributos correctamente."
)

cursor.close()
db.close()

Cargando registros a la base de datos 'laravel-crm'...
¡Proceso finalizado! Se insertaron 793 contactos con sus atributos correctamente.
